In [ ]:
from pathlib import Path
import os
import numpy as np
import pandas as pd

ROOT = Path(os.environ.get("PINPAD_DATA_ROOT", "/home/cse/pinpad/myenv/GNURadio_1/Correlation_Responses/OneSubcarrierLowDataRate"))
OUTPUT = Path(os.environ.get("PINPAD_OUTPUT_ROOT", "/home/cse/pinpad/myenv/GNURadio_1/PINPAD_Results"))
OUTPUT.mkdir(parents=True, exist_ok=True)
N_BUTTONS, N_PER_BUTTON = 12, 200
SOURCE = [(c, "R1") for c in range(1, 6)]
ADAPT_CC = [(c, "R1") for c in range(6, 11)]
ADAPT_CR = [(c, "R2") for c in range(1, 6)]
ADAPT_CCR = [(c, "R2") for c in range(11, 16)]
TEST_CC = [(c, "R1") for c in range(16, 26)]
TEST_CR = [(c, "R3") for c in range(1, 6)]
TEST_CCR = [(c, "R2") for c in range(26, 36)]
ADAPT = {"Source": SOURCE, "CC": ADAPT_CC, "CR": ADAPT_CR, "CCR": ADAPT_CCR}
TEST = {"CC": TEST_CC, "CR": TEST_CR, "CCR": TEST_CCR}
assert sum(map(len, ADAPT.values())) == 20
assert sum(map(len, TEST.values())) == 25
assert not set(sum(ADAPT.values(), [])) & set(sum(TEST.values(), []))
assert 20 * 12 * 200 == 48000 and 25 * 12 * 200 == 60000

COLLECTION_MAP = ROOT / "collection_paths.csv"

def _paths():
    if COLLECTION_MAP.exists():
        frame = pd.read_csv(COLLECTION_MAP)
        if not {"card", "reader", "directory"} <= set(frame.columns):
            raise ValueError("collection_paths.csv requires card,reader,directory")
        mapping = {}
        for row in frame.itertuples(index=False):
            key = (int(row.card), str(row.reader).upper())
            if key in mapping:
                raise ValueError(f"Duplicate collection: {key}")
            p = Path(str(row.directory))
            mapping[key] = p if p.is_absolute() else ROOT / p
        return mapping
    mapping = {}
    for reader in ("R1", "R2", "R3"):
        for card in range(1, 36):
            candidates = [ROOT / reader / f"card{card}", ROOT / reader.lower() / f"card{card}",
                          ROOT / f"card{card}" / reader, ROOT / f"card{card}" / reader.lower()]
            matches = [x for x in candidates if x.is_dir()]
            if len(matches) == 1:
                mapping[(card, reader)] = matches[0]
            elif len(matches) > 1:
                raise ValueError(f"Ambiguous collection paths for {card}, {reader}: {matches}")
    return mapping

COLLECTIONS = _paths()
EXPECTED = set(sum(ADAPT.values(), []) + sum(TEST.values(), []))
missing = sorted(EXPECTED - set(COLLECTIONS))
if missing:
    template = OUTPUT / "collection_paths_REQUIRED.csv"
    pd.DataFrame([{"card": c, "reader": r, "directory": ""} for c, r in sorted(EXPECTED)]).to_csv(template, index=False)
    raise FileNotFoundError(f"Missing {len(missing)} card-reader collections: {missing}. Set PINPAD_DATA_ROOT or fill {COLLECTION_MAP} using {template}; reader identity cannot be guessed from card folder names.")

def _read_collection(key):
    p = COLLECTIONS[key]
    parts = []
    for b in range(N_BUTTONS):
        paths = [p / f"button{b}.npy", p / f"Button{b}.npy"]
        hits = [x for x in paths if x.is_file()]
        if len(hits) != 1:
            raise FileNotFoundError(f"Expected one button{b}.npy in {p}; found {hits}")
        x = np.load(hits[0], allow_pickle=False)
        if len(x) != N_PER_BUTTON:
            raise ValueError(f"{hits[0]} has {len(x)} responses; expected {N_PER_BUTTON}")
        if not np.issubdtype(x.dtype, np.number):
            raise TypeError(f"Non-numeric response array: {hits[0]}")
        if x.ndim != 2 or x.shape[0] != N_PER_BUTTON:
            raise ValueError(f"Expected 200 waveform rows: {hits[0]} {x.shape}")
        if not np.isfinite(x).all():
            raise ValueError(f"Non-finite waveform data: {hits[0]}")
        parts.append(x)
    if len({x.shape[1:] for x in parts}) != 1:
        raise ValueError(f"Response shape mismatch: {p}")
    return np.concatenate(parts), np.repeat(np.arange(N_BUTTONS), N_PER_BUTTON)

def load_group(keys):
    data = [_read_collection(k) for k in keys]
    X = np.concatenate([t[0] for t in data]); y = np.concatenate([t[1] for t in data])
    card = np.concatenate([np.full(len(t[1]), k[0]) for k,t in zip(keys,data)])
    reader = np.concatenate([np.full(len(t[1]), k[1]) for k,t in zip(keys,data)])
    return X, y, card, reader

for label, group in {**{f"adapt_{x}":v for x,v in ADAPT.items()}, **{f"test_{x}":v for x,v in TEST.items()}}.items():
    assert len(group) * 2400 in [12000, 24000]
print("Validated collection map:", len(EXPECTED), "collections;", 108000, "expected total responses")

RESPONSE_LENGTH = 3624
SEED = 42


In [ ]:
expected={"adapt_source":12000,"adapt_CC":12000,"adapt_CR":12000,"adapt_CCR":12000,"test_CC":24000,"test_CR":12000,"test_CCR":24000}
rows=[]
for name,keys in {**{f"adapt_{k}":v for k,v in ADAPT.items()},**{f"test_{k}":v for k,v in TEST.items()}}.items():
    x,y,c,r=load_group(keys)
    if len(x)!=expected[name]:raise ValueError(f"{name} got {len(x)}, expected {expected[name]}")
    rows.append({"domain":name,"collections":len(keys),"samples":len(x),"buttons":len(np.unique(y)),"cards":sorted(set(c)),"readers":sorted(set(r))})
report=pd.DataFrame(rows)
assert report.samples.sum()==108000
report.to_csv(OUTPUT/"verified_dataset_collections.csv",index=False)
print(report.to_string(index=False))
